In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

from model_ranking import load_transfer_metric_results, correlation_table

INFO: P [MainThread] 2025-09-10 17:12:26,743 plantseg - Logger configured at initialisation. PlantSeg logger name: plantseg


/g/kreshuk/talks/miniforge3/envs/model-rank-local2/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/g/kreshuk/talks/pytorch-3dunet/pytorch3dunet/unet3d/predictor.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
transfer_metric_abbrevs = {
    "LEEP": "LEEP",
    "NCTI": "NCTI",
    "LogME": "LogME",
    "Hscore": "Hscore",
    "GBC": "GBC",
    "Regularized_Hscore": "RegHscore",
    "Gaussian_LEEP": "NLEEP",
}

In [5]:
base_path = "/g/kreshuk/talks/sampled_features/semantic_segmentation/mitochondria/5k_pixels_sampled/transfer_metric_results/direct_transfer_results"
result_paths = Path(base_path).glob("*.json")

# Collect all dataframes
all_dfs = []

for result_path in result_paths:
    results = load_transfer_metric_results(result_path)
    correlation_scores = results["correlation_scores"]
    idx = 0
    per_target_KT = np.array(correlation_scores["KT"])
    per_target_SP = np.array(correlation_scores["SP"])
    per_target_PE = np.array(correlation_scores["PE"])
    df = correlation_table(per_target_KT, per_target_SP, per_target_PE, targets = ["EPFL", "Hmito", "Rmito", "VNC"])
    transfer_metric_name = '_'.join(result_path.stem.split("_")[2:])
    transfer_metric_abbrev = transfer_metric_abbrevs[transfer_metric_name]
    
    # Remove the Task index level and add transfer_metric index
    df = df.droplevel("Task")
    df["transfer_metric"] = transfer_metric_abbrev
    df = df.set_index("transfer_metric", append=True)
    
    all_dfs.append(df)

# Combine all dataframes
combined_df = pd.concat(all_dfs)

# Reorder the index levels to have transfer_metric first
combined_df = combined_df.reorder_levels(["transfer_metric", "targets"])

print("Combined correlation table for all transfer metrics:")
print(combined_df)

Loaded Transfer metric results from: /g/kreshuk/talks/sampled_features/semantic_segmentation/mitochondria/5k_pixels_sampled/transfer_metric_results/direct_transfer_results/mito_direct_LEEP.json
Experiment: mito_direct_LEEP
Targets: 4 (EPFL, Hmito, Rmito, VNC)
Source models: 15
Total transfers: 57
Loaded Transfer metric results from: /g/kreshuk/talks/sampled_features/semantic_segmentation/mitochondria/5k_pixels_sampled/transfer_metric_results/direct_transfer_results/mito_direct_Hscore.json
Experiment: mito_direct_Hscore
Targets: 4 (EPFL, Hmito, Rmito, VNC)
Source models: 15
Total transfers: 57
Loaded Transfer metric results from: /g/kreshuk/talks/sampled_features/semantic_segmentation/mitochondria/5k_pixels_sampled/transfer_metric_results/direct_transfer_results/mito_direct_NCTI.json
Experiment: mito_direct_NCTI
Targets: 4 (EPFL, Hmito, Rmito, VNC)
Source models: 15
Total transfers: 57
Loaded Transfer metric results from: /g/kreshuk/talks/sampled_features/semantic_segmentation/mitochond

In [6]:
print("\nAll transfer metrics for EPFL target:")
print(combined_df.xs('EPFL', level='targets'))


All transfer metrics for EPFL target:
                   kt  kt pval  s rho  s rho pval    pr  pr pval
transfer_metric                                                 
LEEP             0.96     0.00   0.92        0.00  0.83     0.00
Hscore          -0.00     0.99  -0.20        0.49 -0.09     0.65
NCTI             0.07     0.80  -0.16        0.55 -0.09     0.69
GBC              0.60     0.02   0.52        0.06  0.37     0.05
LogME           -0.30     0.28  -0.34        0.22 -0.26     0.21
RegHscore       -0.00     0.99  -0.20        0.45 -0.09     0.73
NLEEP            0.41     0.13   0.39        0.16  0.28     0.18


In [7]:
print("\nAll transfer metrics for Hmito target:")
print(combined_df.xs('Hmito', level='targets'))


All transfer metrics for Hmito target:
                   kt  kt pval  s rho  s rho pval    pr  pr pval
transfer_metric                                                 
LEEP             0.96     0.00   0.98        0.00  0.90     0.00
Hscore           0.52     0.05   0.60        0.02  0.46     0.02
NCTI             0.42     0.12   0.50        0.06  0.35     0.07
GBC              0.54     0.04   0.41        0.13  0.35     0.08
LogME            0.05     0.86   0.17        0.59  0.10     0.61
RegHscore        0.51     0.05   0.52        0.06  0.36     0.08
NLEEP            0.64     0.01   0.87        0.00  0.71     0.00


In [8]:
print("\nAll transfer metrics for Rmito target:")
print(combined_df.xs('Rmito', level='targets'))


All transfer metrics for Rmito target:
                   kt  kt pval  s rho  s rho pval    pr  pr pval
transfer_metric                                                 
LEEP             0.98     0.00   0.98        0.00  0.92     0.00
Hscore           0.48     0.07   0.57        0.02  0.39     0.06
NCTI             0.36     0.19   0.04        0.91  0.06     0.74
GBC              0.47     0.08   0.52        0.05  0.36     0.07
LogME            0.09     0.76   0.32        0.23  0.23     0.27
RegHscore        0.47     0.08   0.59        0.03  0.41     0.03
NLEEP            0.62     0.01   0.80        0.00  0.62     0.00
